<a href="https://colab.research.google.com/github/FridaOyucho/HTS-Model-/blob/main/Dropping_cols.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

In [ ]:
#Load Clients data
df = pd.read_csv('clients.csv')
#rename the dataframe to Clients
clients = df

In [ ]:
#Print the columns in the dataset
print(clients.columns.tolist())

['PatientPk', 'SiteCode', 'Dob', 'Sex', 'MaritalStatus', 'PatientDisabled']


In [ ]:
#view the cols and rows
rows, cols = clients.shape
print(f"There are {rows} rows and {cols} columns in the clients table")

There are 12199267 rows and 6 columns in the clients table


In [ ]:
#Remove duplicates
clients.drop_duplicates(inplace=True)
print(clients.shape[0]) #No duplicates detencted in the dataset

12199267


In [ ]:
#Convert DoB to datetime
clients['Dob'] = pd.to_datetime(clients['Dob'])

In [ ]:
#Convert sex to uppercase
clients['Sex'] = clients['Sex'].str.upper()
#print(clients['Sex'])
display(clients['Sex'].value_counts(dropna=False))

,count
Sex,
FEMALE,8012615
MALE,4186625
NaN,27


In [ ]:
#Marital Status
clients['MaritalStatus']= clients['MaritalStatus'].str.upper()
display(clients['MaritalStatus'].value_counts(dropna=False))


,count
MaritalStatus,
MARRIED MONOGAMOUS,5220942
SINGLE,3383747
NaN,2282446
UNKNOWN,534768
MARRIED POLYGAMOUS,289409
DIVORCED,230968
WIDOWED,142193
COHABITING,70702
SEPARATED,39990


In [ ]:
#PatientsDisabled
clients['PatientDisabled'] = clients['PatientDisabled'].str.upper()
display(clients['PatientDisabled'].value_counts(dropna=False))

,count
PatientDisabled,
NO,8020718
YES,4167603
NaN,10946


Handle missing values

In [ ]:
clients.isnull().sum()

,0
PatientPk,0
SiteCode,0
Dob,303
Sex,27
MaritalStatus,2282446
PatientDisabled,10946


In [ ]:
#check missingness % for each column
missing = clients.isnull().sum()
missing_percent = (missing / len(clients)) * 100
print(pd.concat([missing, missing_percent], axis=1,
keys=['MissingCount', 'MissingPercent']))

                 MissingCount  MissingPercent
PatientPk                   0        0.000000
SiteCode                    0        0.000000
Dob                       303        0.002484
Sex                        27        0.000221
MaritalStatus         2282446       18.709698
PatientDisabled         10946        0.089727


Prepare the Test Dataset

In [ ]:
#Load the data
test= pd.read_csv('tests.csv')
#print(test.columns.tolist())
print(test.columns)

/tmp/ipykernel_67406/495268893.py:2: DtypeWarning: Columns (24,27) have mixed types. Specify dtype option on import or set low_memory=False.
  test= pd.read_csv('tests.csv')


Index(['FacilityName', 'SiteCode', 'PatientPk', 'Emr', 'Project',
       'EncounterId', 'TestDate', 'EverTestedForHiv', 'MonthsSinceLastTest',
       'ClientTestedAs', 'EntryPoint', 'TestStrategy', 'TestResult1',
       'TestResult2', 'TestResult3', 'FinalTestResult', 'PatientGivenResult',
       'TbScreening', 'ClientSelfTested', 'CoupleDiscordant', 'TestType',
       'Consent', 'Setting', 'Approach', 'HtsRiskCategory', 'HtsRiskScore',
       'PatientPKHash', 'OtherReferredServices', 'ReferredForServices',
       'ReferredServices', 'LoadDate', 'RecordUUID', 'PriorityPopulationType',
       'DateExtracted'],
      dtype='object')


In [ ]:
#check missingness % for each column
missingtest = test.isnull().sum()
missingtest_percent = (missingtest / len(test)) * 100
test_missing_info = pd.concat([missingtest, missingtest_percent], axis=1,
keys=['missingtest', 'missingtest_percent'])
print(test_missing_info)

                        missingtest  missingtest_percent
FacilityName                      0             0.000000
SiteCode                          0             0.000000
PatientPk                         0             0.000000
Emr                               0             0.000000
Project                           0             0.000000
EncounterId                       0             0.000000
TestDate                          0             0.000000
EverTestedForHiv             119680             3.790471
MonthsSinceLastTest         1861143            58.945598
ClientTestedAs                79942             2.531901
EntryPoint                    22186             0.702669
TestStrategy                      0             0.000000
TestResult1                       0             0.000000
TestResult2                       9             0.000285
TestResult3                 3034503            96.107926
FinalTestResult                   0             0.000000
PatientGivenResult             

In [ ]:
#Print columns with more than 50% missingness
test_missing_info[test_missing_info['missingtest_percent'] > 50]

,missingtest,missingtest_percent
MonthsSinceLastTest,1861143,58.945598
TestResult3,3034503,96.107926
CoupleDiscordant,2979652,94.370700
HtsRiskCategory,3157390,99.999968
HtsRiskScore,3157390,99.999968
OtherReferredServices,3157388,99.999905
PriorityPopulationType,3021901,95.708799
DateExtracted,3157391,100.000000


Data Manipulation to understand why these columns have missing values

In [ ]:
#OtherReferredServices
pd.crosstab(
    test['ReferredForServices'],
    test['OtherReferredServices'].isna()
)

##Drop the column
#Despite the strong association between ReferredForServices and OtherReferredservices
#The column is largely emply

OtherReferredServices,False,True
ReferredForServices,,
No,0,876022
Yes,3,2212751


In [ ]:
#MonthsSincelastTest
#Crosstab with EverTestedforHIV
pd.crosstab(
    test['EverTestedForHiv'],
    test['MonthsSinceLastTest'].isna()
)
## Keep the column
# There is an association between EverTestedForHIV and MonthsSincelastTest.
# Data is largely missing for clients who answered 'No' for Evertestedfor HIV because
#  MonthssincelastTested only applies for the clients who answerd 'Yes'

MonthsSinceLastTest,False,True
EverTestedForHiv,,
No,33108,1369107
Yes,1259435,376061


In [ ]:
#CoupleDiscordant
pd.crosstab(
    test['ClientTestedAs'],
    test['CoupleDiscordant'].isna()
)
## Keep the column.
# There is an association between ClientTestedAs and CoupleDiscordant.
# Data is largely missing for clients tested as Individuals because the question
# of discordant only applies for couples.For individuals this information is not collected

CoupleDiscordant,False,True
ClientTestedAs,,
Couple,85917,43805
Individual,81210,2866517


In [ ]:
#TestResult3
pd.crosstab(
    test['FinalTestResult'],
    test['TestResult3'].isna()
)

#Keep Column
#TestResult3 is only recorded if FinalTestResult is either Inconclusive or Positive
#It is not missing at random

TestResult3,False,True
FinalTestResult,,
Inconclusive,290,1535
Invalid,0,3
Negative,454,3023575
Positive,122144,9390


In [ ]:
#Check where testresult3 is missing but Test1&2 are not same
#All good because testresult1 is mostly negative
problem_records = test[
    (test['TestResult1'] != test['TestResult2']) &
    (test['TestResult3'].isna())
]

problem_records[
    ['PatientPk','TestDate','TestResult1',
     'TestResult2','TestResult3',
     'FinalTestResult']
].head()

,PatientPk,TestDate,TestResult1,TestResult2,TestResult3,FinalTestResult
0,3282,2025-04-07 00:00:00.0000000,Negative,empty,NaN,Negative
1,1737,2025-12-09 00:00:00.0000000,Negative,empty,NaN,Negative
2,3283,2025-04-22 00:00:00.0000000,Negative,empty,NaN,Negative
3,16502,2025-03-26 00:00:00.0000000,Negative,empty,NaN,Negative
4,17258,2025-03-14 00:00:00.0000000,Negative,empty,NaN,Negative


In [ ]:
pd.crosstab(
    test['TestResult3'],
    test['FinalTestResult'],
    margins=True
)

FinalTestResult,Inconclusive,Negative,Positive,All
TestResult3,,,,
Negative,270,41,200,511
Positive,20,413,121944,122377
All,290,454,122144,122888


In [ ]:
pd.crosstab(
    test['TestResult1'] != test['TestResult2'],
    test['TestResult3'].notna(),
    margins=True
)

TestResult3,False,True,All
row_0,,,
False,42732,121663,164395
True,2991771,1225,2992996
All,3034503,122888,3157391


In [ ]:
discordant = (
    test['TestResult1'].notna() &
    test['TestResult2'].notna() &
    (test['TestResult1'] != test['TestResult2'])
)


In [3]:
#pd.crosstab(
    #[df['TestResult1'], df['TestResult2']],
    #df['TestResult3'].notna(),
    #margins=True
#)

Prepare Eligibility data

In [3]:
#Load data

eligibility = pd.read_csv('eligibility.csv')
rows,cols = eligibility.shape
print(f"There are {rows} rows and {cols} columns in the eligibility table")

/tmp/ipykernel_26950/1904812638.py:3: DtypeWarning: Columns (24,68,83) have mixed types. Specify dtype option on import or set low_memory=False.
  eligibility = pd.read_csv('eligibility.csv')


There are 2709027 rows and 87 columns in the eligibility table


In [4]:
#View colums
print(eligibility.columns)


Index(['FacilityName', 'SiteCode', 'PatientPk', 'HtsNumber', 'Emr', 'Project',
       'Processed', 'QueueId', 'Status', 'StatusDate', 'EncounterId',
       'VisitID', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'Department', 'PatientType', 'IsHealthWorker',
       'RelationshipWithContact', 'TestedHIVBefore', 'WhoPerformedTest',
       'ResultOfHIV', 'DateTestedSelf', 'StartedOnART', 'CCCNumber',
       'EverHadSex', 'SexuallyActive', 'NewPartner', 'PartnerHIVStatus',
       'CoupleDiscordant', 'MultiplePartners', 'NumberOfPartners',
       'AlcoholSex', 'MoneySex', 'CondomBurst', 'UnknownStatusPartner',
       'KnownStatusPartner', 'Pregnant', 'BreastfeedingMother',
       'ExperiencedViolenceScreening', 'ContactWithTBCase', 'Lethargy',
       'EverOnPrep', 'CurrentlyOnPrep', 'EverOnPep', 'CurrentlyOnPep',
       'EverHadSTI', 'CurrentlyHasSTI', 'EverHadTB', 'SharedNeedle',
       'NeedleStickInjuries', 'TraditionalProcedures',
       'ChildReasonsForI

In [5]:
#Check for missingness
missingness= eligibility.isnull().sum()
Missingness_percentage = missingness/len(eligibility)
eligibility_missing_info = pd.concat([missingness, Missingness_percentage], axis=1,
keys=['missingness', 'missingness_percentage'])
print(eligibility_missing_info)


                   missingness  missingness_percentage
FacilityName                 0                0.000000
SiteCode                     0                0.000000
PatientPk                    0                0.000000
HtsNumber                  260                0.000096
Emr                          0                0.000000
...                        ...                     ...
ReasonNotReffered      2508276                0.925896
HtsRiskScore            856911                0.316317
LoadDate                     0                0.000000
DateExtracted          2709027                1.000000
RecordUUID                   0                0.000000

[87 rows x 2 columns]


In [6]:
#Quick Check
#Columns with >50% missingness

eligibility_missing_info[eligibility_missing_info['missingness_percentage'] > 0.5]

,missingness,missingness_percentage
Processed,2709027,1.000000
QueueId,2709027,1.000000
Status,2709027,1.000000
StatusDate,2709027,1.000000
KeyPopulation,2490906,0.919484
PriorityPopulation,2630705,0.971089
RelationshipWithContact,2451037,0.904767
DateTestedSelf,2681107,0.989694
StartedOnART,2706738,0.999155
CCCNumber,2708419,0.999776


In [9]:
#We drop variables with 100% missingness and have low predictive power value and some are outcomes of screening

cols_to_drop = ['Processed','QueueId','CCCNumber','DateCreated','DateLastModified',
                            'DateExtracted','StatusDate','Status','EverOnPrep',
                            'EverOnPep','EverHadSTI','EverHadTB','EncounterId','RecordUUID','HIVRiskCategory','HtsRiskScore']

eligibility = eligibility.drop(columns=cols_to_drop, errors='ignore')

In [10]:
print(eligibility.columns)

Index(['FacilityName', 'SiteCode', 'PatientPk', 'HtsNumber', 'Emr', 'Project',
       'VisitID', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'Department', 'PatientType', 'IsHealthWorker',
       'RelationshipWithContact', 'TestedHIVBefore', 'WhoPerformedTest',
       'ResultOfHIV', 'DateTestedSelf', 'StartedOnART', 'EverHadSex',
       'SexuallyActive', 'NewPartner', 'PartnerHIVStatus', 'CoupleDiscordant',
       'MultiplePartners', 'NumberOfPartners', 'AlcoholSex', 'MoneySex',
       'CondomBurst', 'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'ContactWithTBCase', 'Lethargy', 'CurrentlyOnPrep', 'CurrentlyOnPep',
       'CurrentlyHasSTI', 'SharedNeedle', 'NeedleStickInjuries',
       'TraditionalProcedures', 'ChildReasonsForIneligibility',
       'EligibleForTest', 'ReasonsForIneligibility',
       'SpecificReasonForIneligibility', 'Cough', 'DateTestedProvider',
       'Fev

In [ ]:
#We keep all the other records

In [14]:
#Check for variance among numerical variables:
cols = [
    'StartedOnART',
    'CurrentlyOnPrep',
    'CurrentlyOnPep',
    'SharedNeedle',
    'NeedleStickInjuries',
    'ForcedSex',
    'ContactWithTBCase',
    'Lethargy',
    'NightSweats',
    'WeightLoss',
    'Fever',
    'Cough',
    'CurrentlyHasSTI',
    'Pregnant',
    'BreastfeedingMother',
    'AlcoholSex',
    'MoneySex',
    'CondomBurst'
]

for col in cols:
    print(f"\n{'='*40}")
    print(f"Column: {col}")
    print(eligibility[col].value_counts(dropna=False, normalize=True) * 100)



Column: StartedOnART
StartedOnART
NaN    99.915505
Yes     0.067220
No      0.017276
Name: proportion, dtype: float64

Column: CurrentlyOnPrep
CurrentlyOnPrep
NO                    50.993401
NaN                   45.135135
YES                    3.278557
Declined to answer     0.331115
No                     0.197783
Yes                    0.064008
Name: proportion, dtype: float64

Column: CurrentlyOnPep
CurrentlyOnPep
NaN    95.781733
NO      4.122144
YES     0.096123
Name: proportion, dtype: float64

Column: SharedNeedle
SharedNeedle
NaN    98.534824
No      1.446903
Yes     0.018272
Name: proportion, dtype: float64

Column: NeedleStickInjuries
NeedleStickInjuries
NaN    96.882460
No      3.109234
Yes     0.008306
Name: proportion, dtype: float64

Column: ForcedSex
ForcedSex
NaN    99.943633
No      0.050498
Yes     0.005869
Name: proportion, dtype: float64

Column: ContactWithTBCase
ContactWithTBCase
No     96.823361
NaN     3.176639
Name: proportion, dtype: float64

Column: Lethar

In [ ]:
#Have very low variance but will not drop because they are clinically important
#started on ART,currently on pep,sharedneedles,
#needlestickinjuries,forcedsex, lithergy,contactwithTBCase,nightsweats,weightloss,fever because they have very low variance